# MIT News: scrape article content

Reads `./Data/MIT Articles Links.csv` and extracts date, author, source, headline, sub-heading, and body from every article page using MIT News' stable CSS selectors.

Output: `./Data/MIT Articles.csv` (raw scrape; duplicated/missing bodies are repaired in `MIT_03_Fix_Duplicated_Bodies.ipynb`)


In [ ]:
import random
from pathlib import Path
from time import sleep

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

DATA_DIR = Path("./Data")
LINKS_PATH = DATA_DIR / "MIT Articles Links.csv"
OUTPUT_PATH = DATA_DIR / "MIT Articles.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)

links = pd.read_csv(LINKS_PATH)
links = links.loc[:, ~links.columns.str.startswith("Unnamed:")]
links = links.dropna(subset=["Link"]).drop_duplicates(subset="Link").reset_index(drop=True)
print(f"Articles to scrape: {len(links)}")

In [ ]:
def scrape_article(soup, url):
    record = {"Published Date": "", "Author": "", "Source": "",
              "Article Header": "", "Sub_Headings": "", "Article Body": "", "Url": url}

    header = soup.find("div", {"id": "block-mit-page-title"})
    header = header.find("span") if header else None
    if header:
        record["Article Header"] = header.get_text()

    author = soup.find("span", {"class": "news-article--author"})
    if author:
        record["Author"] = author.get_text()

    source = soup.find("span", {"class": "news-article--source"})
    if source:
        record["Source"] = source.get_text()

    date_container = soup.find("div", {"class": "news-article--publication-date"})
    date_tag = date_container.find("time") if date_container else None
    if date_tag:
        record["Published Date"] = date_tag.get_text()

    body_tag = soup.find(
        "div",
        {"class": "paragraph paragraph--type--content-block-text paragraph--view-mode--default"},
    )
    if body_tag:
        record["Article Body"] = [p.get_text().strip() for p in body_tag.find_all("p")]

    sub_heading = soup.find("div", {"class": "news-article--dek"})
    if sub_heading:
        span = sub_heading.find("span")
        record["Sub_Headings"] = span.get_text() if span else sub_heading.get_text()

    return record

In [ ]:
records = []

for url in tqdm(links["Link"]):
    try:
        response = requests.get(url, timeout=30)
    except requests.exceptions.RequestException as exc:
        print(f"Request failed: {url} -> {exc}")
        continue
    if response.status_code != 200:
        print(f"HTTP {response.status_code}: {url}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")

    body_tag = soup.find("body")
    if body_tag and body_tag.get("class") and body_tag["class"][0] == "page-not-found":
        continue

    records.append(scrape_article(soup, url))
    sleep(random.choice([0.3, 0.5]))

articles = pd.DataFrame(records)
print(f"Scraped {len(articles)} articles.")
articles.info()

In [ ]:
articles.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH}")